<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Hunyuan3D-2.1_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Hunyuan3D-2.1 — Tencent's Production-Ready 3D Asset Pipeline

A self-contained Colab wrapper around [Tencent's Hunyuan3D-2.1](https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1) — the production successor to Hunyuan3D-2.0. Given a single image, generate a clean, low-poly, **PBR-textured** `.glb` mesh you can drop into a game engine or Three.js and have real metallic / roughness / subsurface-scattering lighting respond correctly.

- **Shape generation (3.3 B DiT, v2-1)** — flow-matching transformer that turns a single RGB image into a watertight mesh. Significantly higher quality than 2.0's 1.1 B model. ~6.5 GB on disk, ~10 GB VRAM.
- **PBR texture synthesis (2 B Hunyuan3D-Paint-2.1)** — multi-view diffusion pipeline that generates **albedo + metallic + roughness** maps, then bakes them into a single UV-mapped texture. This is the key upgrade over 2.0: textures respond to lighting realistically. ~4 GB on disk, ~21 GB VRAM.
- **Mesh cleanup** — `FloaterRemover`, `DegenerateFaceRemover`, and `FaceReducer` keep the output low-poly and printable.
- **Texture an existing mesh** — upload any `.glb / .obj / .ply` + a reference image, get it re-textured with PBR. Useful for re-texturing outputs from Cube 3D, CubePart, or anything else.

All weights are pulled from [tencent/Hunyuan3D-2.1](https://huggingface.co/tencent/Hunyuan3D-2.1) on Hugging Face and cached on your Google Drive so subsequent runs are instant. The notebook is based on the official [Tencent Hunyuan3D-2.1 Gradio app](https://huggingface.co/spaces/tencent/Hunyuan3D-2.1) and adds Colab-friendly 7-step ergonomics.

**Disclaimer:** The model weights are released under the [Tencent Hunyuan Community License](https://huggingface.co/tencent/Hunyuan3D-2.1/blob/main/LICENSE.txt) (non-commercial research use). Generated assets are not moderated by Tencent safety systems. You are responsible for downstream use.

**GPU note:** The full pipeline (shape + PBR) needs ~29 GB VRAM (A100 or L4 with careful offload). On a 16 GB T4 you can run the shape pipeline (3.3 B) but the PBR paint pipeline will OOM. The UI will tell you so and gracefully fall back to white-mesh only.

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Clones the [Tencent-Hunyuan/Hunyuan3D-2.1](https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1) repo,
# @markdown builds the `custom_rasterizer` + `mesh_painter` CUDA extensions,
# @markdown downloads the RealESRGAN x4 upscaler checkpoint, and applies the torchvision
# @markdown compatibility patch needed by `basicsr` (used inside the paint pipeline).
# @markdown First run: ~5-8 min. Subsequent runs: <30 s (skip when already installed).

import os, sys, subprocess, time, shutil, pathlib, importlib

REPO_DIR = '/content/Hunyuan3D-2.1'
REPO_URL = 'https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git'

if not os.path.isdir(REPO_DIR):
    print(f'[Install] Cloning {REPO_URL} ...')
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, REPO_DIR], check=True)
    print(f'[Install] Cloned to {REPO_DIR}')
else:
    print(f'[Install] {REPO_DIR} already present.')

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, 'hy3dshape'))
    sys.path.insert(0, os.path.join(REPO_DIR, 'hy3dpaint'))
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

t0 = time.time()
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                 'ninja==1.11.1.1', 'pybind11==2.13.4', 'gradio==5.33.0',
                 'transformers==4.46.0', 'diffusers==0.30.0', 'accelerate==1.1.1',
                 'huggingface-hub>=0.25.0', 'safetensors==0.4.4',
                 'numpy<2.0', 'scipy', 'einops==0.8.0', 'opencv-python',
                 'trimesh==4.4.7', 'pymeshlab==2022.2.post3', 'pygltflib==1.16.3',
                 'xatlas==0.0.9', 'rembg==2.0.65', 'realesrgan==0.3.0', 'basicsr==1.4.2',
                 'omegaconf==2.3.0', 'pyyaml==6.0.2', 'tqdm==4.66.5', 'psutil',
                 'fastapi==0.115.12', 'uvicorn==0.34.3', 'onnxruntime', 'timm',
                 'tb_nightly', 'scikit-image', 'imageio', 'pandas'], check=True)
print(f'[Install] pip install took {time.time() - t0:.1f}s')

# Apply torchvision compatibility fix BEFORE importing basicsr (used by paint pipeline).
def _apply_torchvision_fix():
    try:
        spec = importlib.util.spec_from_file_location(
            'torchvision_fix', os.path.join(REPO_DIR, 'torchvision_fix.py'))
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        mod.apply_fix()
        return True, 'patched'
    except Exception as e:
        return False, str(e)

fix_ok, fix_msg = _apply_torchvision_fix()
print(f'[Install] torchvision fix: {fix_ok} ({fix_msg})')

# Build the CUDA extensions. Graceful fallback to 'disabled paint' if nvcc is missing.
def _try_build_cuda_extensions():
    from torch.utils.cpp_extension import CUDA_HOME
    if not CUDA_HOME or not os.path.exists(os.path.join(CUDA_HOME, 'bin', 'nvcc')):
        return 'no-nvcc', 'No nvcc found; PBR texture synthesis will be disabled.'
    log = []
    # 1) custom_rasterizer (custom CUDA kernels used by paint rasterizer)
    t0 = time.time()
    try:
        subprocess.run([sys.executable, 'setup.py', 'install'],
                       cwd=os.path.join(REPO_DIR, 'hy3dpaint', 'custom_rasterizer'),
                       check=True, capture_output=True)
        log.append(f'custom_rasterizer OK ({time.time() - t0:.0f}s)')
    except subprocess.CalledProcessError as e:
        return 'rasterizer-failed', f'custom_rasterizer build failed: {(e.stderr or b"")[-500:].decode(errors="ignore")}'
    # 2) DifferentiableRenderer / mesh_inpaint_processor (C++ only)
    t0 = time.time()
    dr_dir = os.path.join(REPO_DIR, 'hy3dpaint', 'DifferentiableRenderer')
    try:
        compile_script = os.path.join(dr_dir, 'compile_mesh_painter.sh')
        if os.path.exists(compile_script):
            subprocess.run(['bash', compile_script], cwd=dr_dir, check=True, capture_output=True)
        else:
            subprocess.run(['c++', '-O3', '-Wall', '-shared', '-std=c++11', '-fPIC',
                            '$(python -m pybind11 --includes)', 'mesh_inpaint_processor.cpp',
                            '-o', f'mesh_inpaint_processor{sysconfig.get_config_var("EXT_SUFFIX")}'],
                           cwd=dr_dir, check=True, capture_output=True, shell=True)
        log.append(f'mesh_inpaint_processor OK ({time.time() - t0:.0f}s)')
    except subprocess.CalledProcessError as e:
        log.append(f'mesh_inpaint_processor warning: {(e.stderr or b"")[-200:].decode(errors="ignore")}')
    return 'ok', ' ; '.join(log)

import sysconfig  # for EXT_SUFFIX in fallback compile command
print('[Install] Building CUDA extensions (may take a couple of minutes) ...')
rasterizer_status, rasterizer_msg = _try_build_cuda_extensions()
print(f'[Install] CUDA build: {rasterizer_status} — {rasterizer_msg}')

# Download RealESRGAN x4 upscaler (~65 MB) used by the paint pipeline.
import urllib.request
ckpt_dir = pathlib.Path(REPO_DIR) / 'hy3dpaint' / 'ckpt'
ckpt_dir.mkdir(parents=True, exist_ok=True)
realesrgan_path = ckpt_dir / 'RealESRGAN_x4plus.pth'
if not realesrgan_path.exists() or realesrgan_path.stat().st_size < 60_000_000:
    url = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
    print(f'[Install] Downloading RealESRGAN x4plus to {realesrgan_path} ...')
    urllib.request.urlretrieve(url, str(realesrgan_path))
    print(f'[Install] RealESRGAN downloaded: {realesrgan_path.stat().st_size/1024/1024:.0f} MB')
else:
    print(f'[Install] RealESRGAN already present ({realesrgan_path.stat().st_size/1024/1024:.0f} MB).')

# Sanity-check imports.
try:
    from hy3dshape import (Hunyuan3DDiTFlowMatchingPipeline, FaceReducer,
                           FloaterRemover, DegenerateFaceRemover)
    from hy3dshape.rembg import BackgroundRemover
    print('[Install] hy3dshape OK')
except Exception as e:
    raise SystemExit(f'hy3dshape import failed: {e}')

try:
    from hy3dpaint.textureGenPipeline import Hunyuan3DPaintPipeline, Hunyuan3DPaintConfig
    print('[Install] hy3dpaint OK')
    _PAINT_IMPORT_OK = True
except Exception as e:
    print(f'[Install] hy3dpaint import failed: {e}')
    Hunyuan3DPaintPipeline = Hunyuan3DPaintConfig = None
    _PAINT_IMPORT_OK = False

HAS_TEXTUREGEN = (rasterizer_status == 'ok') and _PAINT_IMPORT_OK
if not HAS_TEXTUREGEN:
    print('[Install] PBR texture synthesis will be disabled in the UI.')

print(f'\n[Install] DONE in {time.time() - t0:.1f}s total.')
print(f'[Install] GPU: {__import__("torch").cuda.get_device_name(0)}')

In [ ]:
# @title STEP 2 — Cache Weights & Example Images
# @markdown Downloads the 2.1 shape + PBR paint weights to Google Drive (~10 GB total)
# @markdown and fetches 6 example images from the upstream 2.0 repo (the 2.1 example set is the same).

import os, sys, time, shutil, pathlib, requests
from huggingface_hub import snapshot_download, hf_hub_download
from tqdm.notebook import tqdm

CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_TTS_Cache/huggingface/hub')
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE_ROOT.parent)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE_ROOT)
print(f'[Cache] HF cache root: {CACHE_ROOT}')

ASSETS_DIR = pathlib.Path('/content/hy3d21_assets')
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
EXAMPLE_IMAGES = ['052.png', '073.png', '1008.png', '1022.png', '1146.png', '1180.png']

MODEL_REGISTRY = {
    'shape-2.1': ('tencent/Hunyuan3D-2.1', ['hunyuan3d-dit-v2-1']),
    'paint-2.1': ('tencent/Hunyuan3D-2.1', ['hunyuan3d-paintpbr-v2-1']),
}

def _human(n):
    for u in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024: return f'{n:.1f} {u}'
        n /= 1024
    return f'{n:.1f} PB'

def _dir_size(p):
    p = pathlib.Path(p)
    if not p.exists(): return 0
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())

def download_component(name, force=False):
    if name not in MODEL_REGISTRY:
        raise ValueError(f'unknown component: {name}')
    repo_id, subfolders = MODEL_REGISTRY[name]
    local_dir = CACHE_ROOT / repo_id.replace('/', '_')
    existing = _dir_size(local_dir)
    if existing > 200_000_000 and not force:
        print(f'[Cache] {name} already cached ({_human(existing)}). Skipping.')
        return str(local_dir)
    print(f'[Cache] Downloading {name} from {repo_id} (subfolders: {subfolders}) ...')
    snapshot_download(
        repo_id=repo_id,
        allow_patterns=[f'{sf}/*' for sf in subfolders] + ['*.json', '*.txt'],
        local_dir=str(local_dir),
        local_dir_use_symlinks=False,
        tqdm_class=tqdm,
    )
    print(f'[Cache] {name} now on disk: {_human(_dir_size(local_dir))}')
    return str(local_dir)

def fetch_examples(force=False):
    got = []
    for name in EXAMPLE_IMAGES:
        dst = ASSETS_DIR / name
        if dst.exists() and dst.stat().st_size > 1000 and not force:
            got.append(str(dst)); continue
        url = f'https://raw.githubusercontent.com/Tencent-Hunyuan/Hunyuan3D-2.1/main/assets/example_images/{name}'
        try:
            r = requests.get(url, timeout=20)
            r.raise_for_status()
            dst.write_bytes(r.content)
            got.append(str(dst))
        except Exception as e:
            print(f'  [Example] {name}: {e}')
    print(f'[Cache] {len(got)}/{len(EXAMPLE_IMAGES)} example images in {ASSETS_DIR}')
    return got

# Edit COMPONENTS to add/remove downloads. Default = both (full pipeline).
# On a 16 GB T4 you can drop 'paint-2.1' to save ~4 GB of disk and skip the OOM-prone pipeline.
COMPONENTS = ['shape-2.1', 'paint-2.1']

if shutil.disk_usage('/content').free < 30 * 1024**3:
    print('[Cache] WARNING: less than 30 GB free on /content. Consider dropping paint-2.1.')

_dirs = {}
for c in COMPONENTS:
    _dirs[c] = download_component(c)

examples = fetch_examples()
print(f'\n[Cache] DONE. Weights in: {list(_dirs.values())}')
print(f'[Cache] Examples in: {ASSETS_DIR}')
print('Run Step 3 next.')

In [ ]:
# @title STEP 3 — Pipeline Wrappers (Lazy Load)
# @markdown Defines lazy-loading wrappers for the 2.1 shape and PBR paint pipelines.
# @markdown Shape loads on first use; paint loads on first PBR request.

import gc, os, sys, time, traceback, pathlib, json, random, importlib
import torch, numpy as np, trimesh
from PIL import Image

REPO_DIR = '/content/Hunyuan3D-2.1'
if REPO_DIR not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, 'hy3dshape'))
    sys.path.insert(0, os.path.join(REPO_DIR, 'hy3dpaint'))
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

# Re-apply the torchvision fix in case the kernel was reset between cells.
try:
    spec = importlib.util.spec_from_file_location(
        'torchvision_fix', os.path.join(REPO_DIR, 'torchvision_fix.py'))
    _fix_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(_fix_mod)
    _fix_mod.apply_fix()
except Exception as e:
    print(f'[Core] torchvision fix: {e}')

from hy3dshape import (Hunyuan3DDiTFlowMatchingPipeline, FaceReducer,
                       FloaterRemover, DegenerateFaceRemover)
from hy3dshape.pipelines import export_to_trimesh
from hy3dshape.rembg import BackgroundRemover

try:
    from hy3dpaint.textureGenPipeline import Hunyuan3DPaintPipeline, Hunyuan3DPaintConfig
    _PAINT_IMPORT_OK = True
except Exception as e:
    print(f'[Core] Paint import failed: {e}')
    Hunyuan3DPaintPipeline = Hunyuan3DPaintConfig = None
    _PAINT_IMPORT_OK = False

class ShapeEngine:
    """3.3 B Hunyuan3D-2.1 shape pipeline. ~10 GB VRAM."""
    def __init__(self):
        self.pipeline = None
        self.floater = FloaterRemover()
        self.degenerate = DegenerateFaceRemover()
        self.reducer = FaceReducer()
        self.rmbg = BackgroundRemover()

    def load(self):
        if self.pipeline is not None:
            print('[Shape] already loaded.'); return
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print('[Shape] Loading tencent/Hunyuan3D-2.1 / hunyuan3d-dit-v2-1 (3.3 B) ...')
        t0 = time.time()
        self.pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
            'tencent/Hunyuan3D-2.1', subfolder='hunyuan3d-dit-v2-1',
            use_safetensors=False,  # 2.1 ships .bin not .safetensors
            device='cuda',
        )
        print(f'[Shape] Loaded in {time.time() - t0:.1f}s. VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

    def unload(self):
        if self.pipeline is not None:
            print('[Shape] Unloading ...')
            del self.pipeline; self.pipeline = None
            gc.collect(); torch.cuda.empty_cache()

    def generate(self, image_pil, num_inference_steps, guidance_scale, seed, octree_resolution, num_chunks):
        if self.pipeline is None:
            raise RuntimeError('shape engine not loaded')
        g = torch.Generator().manual_seed(int(seed))
        out = self.pipeline(
            image=image_pil,
            num_inference_steps=int(num_inference_steps),
            guidance_scale=float(guidance_scale),
            generator=g,
            octree_resolution=int(octree_resolution),
            num_chunks=int(num_chunks),
            output_type='mesh',
        )
        return export_to_trimesh(out)[0]

    def cleanup(self, mesh):
        mesh = self.floater(mesh)
        mesh = self.degenerate(mesh)
        mesh = self.reducer(mesh)
        return mesh

class PaintEngine:
    """2 B Hunyuan3D-2.1 PBR paint pipeline. ~21 GB VRAM."""
    def __init__(self):
        if not _PAINT_IMPORT_OK:
            self.disabled_reason = 'Paint import failed (likely missing custom_rasterizer).'
        else:
            self.disabled_reason = None
        self.pipeline = None

    def load(self):
        if not _PAINT_IMPORT_OK:
            raise RuntimeError(self.disabled_reason)
        if self.pipeline is not None:
            print('[Paint] already loaded.'); return
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print('[Paint] Loading Hunyuan3DPaintPipeline (2 B PBR) ...')
        t0 = time.time()
        conf = Hunyuan3DPaintConfig(max_num_view=6, resolution=512)
        conf.realesrgan_ckpt_path = os.path.join(REPO_DIR, 'hy3dpaint', 'ckpt', 'RealESRGAN_x4plus.pth')
        conf.multiview_cfg_path = os.path.join(REPO_DIR, 'hy3dpaint', 'cfgs', 'hunyuan-paint-pbr.yaml')
        conf.custom_pipeline = os.path.join(REPO_DIR, 'hy3dpaint', 'hunyuanpaintpbr')
        self.pipeline = Hunyuan3DPaintPipeline(conf)
        print(f'[Paint] Loaded in {time.time() - t0:.1f}s.')

    def unload(self):
        if self.pipeline is not None:
            del self.pipeline; self.pipeline = None
            gc.collect(); torch.cuda.empty_cache()

    def texture(self, mesh_path, image):
        if self.pipeline is None:
            raise RuntimeError('paint engine not loaded')
        # Returns the path to the textured .obj (also creates a sibling .glb).
        return self.pipeline(mesh_path=mesh_path, image_path=image, save_glb=True)

SHAPE = ShapeEngine()
PAINT = PaintEngine()

def _strip_bg(image, do_rembg):
    if image is None: return None
    if image.mode != 'RGBA': image = image.convert('RGBA')
    if do_rembg or image.mode == 'RGB':
        return SHAPE.rmbg(image.convert('RGB'))
    return image

def gen_shape(image, steps, cfg, seed, octree, num_chunks, do_rembg=True):
    if image is None: raise ValueError('image is required')
    image = _strip_bg(image, do_rembg)
    mesh = SHAPE.generate(image, steps, cfg, seed, octree, num_chunks)
    mesh = SHAPE.cleanup(mesh)
    return mesh, image

def gen_textured(mesh_path, image):
    if not _PAINT_IMPORT_OK:
        raise RuntimeError('PBR Paint pipeline not available (custom_rasterizer missing).')
    return PAINT.texture(mesh_path, image)

print('[Core] Engines ready.')
print(f'[Core]   PBR Paint available: {_PAINT_IMPORT_OK}')
print('[Core]   Run Step 4 to launch the UI.')

In [ ]:
# @title STEP 4 — Launch Gradio UI
# @markdown Opens a Gradio app with one tab per operation + a VRAM tab.

import os, sys, time, json, uuid, shutil, pathlib, gc, traceback, random
import torch, numpy as np, trimesh, gradio as gr
from PIL import Image
from IPython.display import clear_output, display, HTML

REPO_DIR = '/content/Hunyuan3D-2.1'
if REPO_DIR not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, 'hy3dshape'))
    sys.path.insert(0, os.path.join(REPO_DIR, 'hy3dpaint'))
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

SAVE_DIR = pathlib.Path('/content/hy3d21_runs')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
ASSETS_DIR = pathlib.Path('/content/hy3d21_assets')
EXAMPLE_PATHS = sorted(str(p) for p in ASSETS_DIR.glob('*.png')) if ASSETS_DIR.exists() else []

def _new_run_dir():
    p = SAVE_DIR / f'run_{int(time.time())}_{uuid.uuid4().hex[:6]}'
    p.mkdir(parents=True, exist_ok=True)
    return p

def _save_mesh(mesh, folder, name='white_mesh.glb', textured=False):
    p = folder / name
    mesh.export(str(p), include_normals=textured)
    return str(p)

def _model_viewer_html(folder, height=620, textured=False, pbr=False):
    fname = 'textured_mesh.glb' if textured else 'white_mesh.glb'
    rel = f'./{fname}'
    badge = '<div style="position:absolute;top:8px;right:8px;background:#7c3aed;color:white;padding:2px 8px;border-radius:4px;font-size:11px;">PBR</div>' if pbr else ''
    return f"""
    <div style='position:relative;height: {height}px; width: 100%; border-radius: 8px; border: 1px solid #e5e7eb;'>
      {badge}
      <model-viewer src="{rel}/" alt="3D mesh" auto-rotate camera-controls
        style='height: {height-2}px; width: 100%;'
        shadow-intensity='0.9' environment-image='neutral'></model-viewer>
    </div>
    <script type='module' src='https://ajax.googleapis.com/ajax/libs/model-viewer/3.5.0/model-viewer.min.js'></script>
    """

def do_generate_shape(image, steps, cfg, seed, octree, num_chunks, do_rembg,
                      randomize_seed, progress=gr.Progress()):
    try:
        SHAPE.load()
        if randomize_seed:
            seed = random.randint(0, 1_000_000)
        progress(0.05, desc='Generating shape (3.3 B DiT) ...')
        t0 = time.time()
        mesh, proc_img = gen_shape(image, int(steps), float(cfg), int(seed), int(octree), int(num_chunks), bool(do_rembg))
        progress(0.85, desc='Saving mesh ...')
        folder = _new_run_dir()
        path = _save_mesh(mesh, folder)
        html = _model_viewer_html(folder)
        stats = {'seed': int(seed), 'faces': int(mesh.faces.shape[0]),
                 'vertices': int(mesh.vertices.shape[0]), 'elapsed_s': round(time.time() - t0, 1)}
        return html, path, proc_img, stats, int(seed)
    except Exception as e:
        traceback.print_exc()
        return None, None, image, {'error': str(e)}, int(seed)

def do_generate_textured(image, steps, cfg, seed, octree, num_chunks, do_rembg,
                         randomize_seed, progress=gr.Progress()):
    try:
        if not _PAINT_IMPORT_OK:
            raise RuntimeError('PBR Paint pipeline not available. Check Step 1 for CUDA build errors.')
        SHAPE.load()
        if randomize_seed:
            seed = random.randint(0, 1_000_000)
        progress(0.05, desc='Generating shape (3.3 B) ...')
        t0 = time.time()
        mesh, proc_img = gen_shape(image, int(steps), float(cfg), int(seed), int(octree), int(num_chunks), bool(do_rembg))
        progress(0.50, desc='Saving white mesh (paint needs .obj) ...')
        folder = _new_run_dir()
        # Paint pipeline needs an .obj path. Save white as .obj for the paint to consume.
        white_obj = folder / 'white_mesh.obj'
        mesh.export(str(white_obj), include_normals=False)
        progress(0.60, desc='Loading PBR paint (2 B) ...')
        PAINT.load()
        progress(0.70, desc='Painting PBR textures (multi-view diffusion) ...')
        textured_obj = gen_textured(str(white_obj), proc_img)
        # The paint pipeline produces <obj> and a sibling <obj>.glb (PBR).
        textured_glb = pathlib.Path(textured_obj).with_suffix('.glb')
        progress(0.95, desc='Saving ...')
        html_white = _model_viewer_html(folder, textured=False)
        html_tex = _model_viewer_html(folder, textured=True, pbr=True)
        stats = {'seed': int(seed),
                 'faces_white': int(mesh.faces.shape[0]),
                 'elapsed_s': round(time.time() - t0, 1),
                 'pbr_output': str(textured_glb) if textured_glb.exists() else str(textured_obj)}
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return html_white, html_tex, str(white_obj), str(textured_glb if textured_glb.exists() else textured_obj), proc_img, stats, int(seed)
    except Exception as e:
        traceback.print_exc()
        return None, None, None, None, image, {'error': str(e)}, int(seed)

def do_texture_existing(mesh_file, image, progress=gr.Progress()):
    try:
        if mesh_file is None:
            raise ValueError('Please upload a .glb / .obj / .ply mesh.')
        if image is None:
            raise ValueError('Please provide the image to paint from.')
        PAINT.load()
        progress(0.1, desc='Loading mesh ...')
        # Paint needs a path. Save the upload to a stable .obj path.
        folder = _new_run_dir()
        mesh = trimesh.load(mesh_file, force='mesh')
        input_obj = folder / 'input_mesh.obj'
        mesh.export(str(input_obj), include_normals=False)
        progress(0.3, desc='Painting PBR textures ...')
        textured_obj = PAINT.texture(str(input_obj), image)
        textured_glb = pathlib.Path(textured_obj).with_suffix('.glb')
        progress(0.9, desc='Saving ...')
        html = _model_viewer_html(folder, textured=True, pbr=True)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return html, str(input_obj), str(textured_glb if textured_glb.exists() else textured_obj)
    except Exception as e:
        traceback.print_exc()
        return None, None, None

def vram_free_shape():
    SHAPE.unload()
    return f'[VRAM] Shape engine freed. {torch.cuda.memory_allocated()/1024**3:.1f} GB allocated.'

def vram_free_paint():
    PAINT.unload()
    return f'[VRAM] Paint engine freed. {torch.cuda.memory_allocated()/1024**3:.1f} GB allocated.'

def vram_free_all():
    SHAPE.unload(); PAINT.unload(); gc.collect(); torch.cuda.empty_cache()
    return f'[VRAM] All engines freed. {torch.cuda.memory_allocated()/1024**3:.1f} GB allocated.'

def vram_status():
    if not torch.cuda.is_available(): return 'No GPU.'
    a = torch.cuda.memory_allocated() / 1024**3
    r = torch.cuda.memory_reserved() / 1024**3
    t = torch.cuda.get_device_properties(0).total_memory / 1024**3
    return (f'Allocated: {a:.1f} GB\nReserved:  {r:.1f} GB\nTotal:     {t:.1f} GB\n'

            f'Shape loaded: {SHAPE.pipeline is not None}\n'

            f'Paint loaded: {PAINT.pipeline is not None}')

with gr.Blocks(title='Hunyuan3D-2.1 Colab', theme=gr.themes.Base(), analytics_enabled=False) as demo:
    gr.HTML("""
    <div style='font-size: 1.8em; font-weight: bold; text-align: center; margin: 4px 0;'>
      Hunyuan3D-2.1 — Image → PBR-Textured 3D</div>
    <div style='text-align: center; color: #6b7280;'>
      The production-ready successor to Hunyuan3D-2.0. 3.3 B shape DiT + 2 B PBR paint.</div>
    """)
    gr.HTML("""<div style='background: #fef3c7; color: #92400e; padding: 8px 12px; border-radius: 6px;
        margin: 6px 0;'>The full pipeline (Shape + PBR Paint) needs ~29 GB VRAM. <b>Recommended GPU: A100 (40 GB) or L4 (24 GB, tight).</b>
        On a T4 (16 GB), use the Shape-only tab; the PBR Paint tab will OOM.</div>""")
    with gr.Tabs():
        # -- Gen shape tab -------------------------------------------
        with gr.Tab('Shape (white mesh)'):
            with gr.Row():
                with gr.Column(scale=3):
                    image_in = gr.Image(label='Input image', type='pil', image_mode='RGBA', height=280,
                                          info='Single image of an object on a clean background. White or transparent works best.')
                    do_rembg = gr.Checkbox(value=True, label='Remove background',
                                           info='Recommended unless your image already has a clean alpha channel.')
                    with gr.Accordion('Advanced options', open=False):
                        steps = gr.Slider(1, 100, value=30, step=1, label='Inference steps',
                                          info='The 2.1 shape DiT is not step-distilled; 30 is a good default.')
                        cfg = gr.Slider(1.0, 15.0, value=5.5, step=0.1, label='Guidance scale',
                                          info='Higher = stronger prompt adherence but more artifacts.')
                        octree = gr.Slider(64, 512, value=256, step=32, label='Octree resolution',
                                           info='Higher = more detail, more VRAM.')
                        num_chunks = gr.Slider(1000, 200_000, value=8000, step=1000, label='Decode chunks',
                                               info='Lower if you hit VRAM OOM during decoding.')
                        seed = gr.Slider(0, 1_000_000, value=1234, step=1, label='Seed',
                                          info='Different seeds give different shapes from the same input.')
                        randomize_seed = gr.Checkbox(value=False, label='Randomize seed')
                    btn_shape = gr.Button('Generate Shape', variant='primary')
                with gr.Column(scale=4):
                    out_html = gr.HTML("""<div style='height: 560px; width: 100%; border-radius: 8px;
                        border: 1px solid #e5e7eb; display: flex; align-items: center; justify-content: center;
                        color: #8d8d8d;'>No mesh yet.</div>""")
                    with gr.Row():
                        out_file = gr.File(label='Download .glb')
                        proc_image = gr.Image(label='Processed input', type='pil', height=120, interactive=False)
                    out_stats = gr.JSON(label='Run stats')
                with gr.Column(scale=2):
                    if EXAMPLE_PATHS:
                        gr.Examples(examples=[[p] for p in EXAMPLE_PATHS], inputs=[image_in],
                                    label='Example images', examples_per_page=6)
            btn_shape.click(
                do_generate_shape,
                inputs=[image_in, steps, cfg, seed, octree, num_chunks, do_rembg, randomize_seed],
                outputs=[out_html, out_file, proc_image, out_stats, seed],
            )
        # -- Gen textured tab ----------------------------------------
        with gr.Tab('Shape + PBR Texture'):
            gr.HTML("""<div style='background: #ede9fe; color: #5b21b6; padding: 8px 12px; border-radius: 6px;
                margin: 6px 0;'>The <b>PBR</b> (physically-based-rendering) paint generates albedo +
                metallic + roughness maps. The result is a <code>.glb</code> with realistic light interaction
                (metallic reflections, subsurface scattering). VRAM hungry: ~21 GB for the paint pipeline alone.</div>""")
            with gr.Row():
                with gr.Column(scale=3):
                    image_t = gr.Image(label='Input image', type='pil', image_mode='RGBA', height=240,
                                         info='Single image of an object on a clean background. White or transparent works best.')
                    do_rembg_t = gr.Checkbox(value=True, label='Remove background')
                    with gr.Accordion('Advanced options', open=False):
                        steps_t = gr.Slider(1, 100, value=30, step=1, label='Inference steps (shape)',
                                              info='30 is fine for 2.1 — there is no step-distilled variant.')
                        cfg_t = gr.Slider(1.0, 15.0, value=5.5, step=0.1, label='Guidance scale',
                                           info='Higher = stronger prompt adherence but more artifacts.')
                        octree_t = gr.Slider(64, 512, value=256, step=32, label='Octree resolution')
                        num_chunks_t = gr.Slider(1000, 200_000, value=8000, step=1000, label='Decode chunks')
                        seed_t = gr.Slider(0, 1_000_000, value=1234, step=1, label='Seed',
                                            info='Different seeds give different shapes from the same input.')
                        randomize_seed_t = gr.Checkbox(value=False, label='Randomize seed')
                    btn_tex = gr.Button('Generate Shape + PBR Texture', variant='primary')
                with gr.Column(scale=4):
                    with gr.Tabs():
                        with gr.Tab('White mesh'):
                            out_white_html = gr.HTML("""<div style='height: 480px; width: 100%;
                                border: 1px solid #e5e7eb; display: flex; align-items: center;
                                justify-content: center; color: #8d8d8d;'>No mesh yet.</div>""")
                        with gr.Tab('PBR-textured mesh'):
                            out_tex_html = gr.HTML("""<div style='height: 480px; width: 100%;
                                border: 1px solid #e5e7eb; display: flex; align-items: center;
                                justify-content: center; color: #8d8d8d;'>No mesh yet.</div>""")
                    with gr.Row():
                        out_white_file = gr.File(label='White .glb')
                        out_tex_file = gr.File(label='PBR .glb')
                        proc_image_t = gr.Image(label='Processed input', type='pil', height=120, interactive=False)
                    out_stats_t = gr.JSON(label='Run stats')
                with gr.Column(scale=2):
                    if EXAMPLE_PATHS:
                        gr.Examples(examples=[[p] for p in EXAMPLE_PATHS], inputs=[image_t],
                                    label='Example images', examples_per_page=6)
            btn_tex.click(
                do_generate_textured,
                inputs=[image_t, steps_t, cfg_t, seed_t, octree_t, num_chunks_t, do_rembg_t, randomize_seed_t],
                outputs=[out_white_html, out_tex_html, out_white_file, out_tex_file, proc_image_t,
                         out_stats_t, seed_t],
            )
        # -- Texture existing tab -----------------------------------
        with gr.Tab('Texture an existing mesh'):
            gr.HTML("""<div style='background: #f0fdf4; color: #166534; padding: 8px 12px; border-radius: 6px;
                margin: 6px 0;'>Upload a hand-made <code>.glb / .obj / .ply</code> mesh plus an image to
                paint from. The output is a PBR-textured <code>.glb</code>. Useful for re-texturing
                outputs from Cube 3D, CubePart, or any other source.</div>""")
            with gr.Row():
                with gr.Column(scale=3):
                    mesh_in = gr.File(label='Input mesh (.glb / .obj / .ply)', file_types=['.glb', '.obj', '.ply', '.stl'])
                    image_e = gr.Image(label='Image to paint from', type='pil', image_mode='RGBA', height=240,
                                         info='Reference image. The output mesh will be painted to match the colors / materials in this image.')
                    btn_e = gr.Button('Paint PBR Texture', variant='primary')
                with gr.Column(scale=5):
                    out_e_html = gr.HTML("""<div style='height: 520px; width: 100%; border: 1px solid #e5e7eb;
                        display: flex; align-items: center; justify-content: center; color: #8d8d8d;'>
                        No mesh yet.</div>""")
                    with gr.Row():
                        out_e_input = gr.File(label='Input mesh (saved)')
                        out_e_tex = gr.File(label='PBR .glb')
            btn_e.click(do_texture_existing, inputs=[mesh_in, image_e],
                        outputs=[out_e_html, out_e_input, out_e_tex])
        # -- VRAM tab -----------------------------------------------
        with gr.Tab('VRAM'):
            gr.HTML("""<div>Use these buttons to swap pipelines in and out of VRAM. 
                Shape and Paint are independent — load the one you need, free the one you don't.</div>""")
            vram_text = gr.Textbox(label='Status', value=vram_status, interactive=False, lines=6)
            with gr.Row():
                free_shape_btn = gr.Button('Free shape engine')
                free_paint_btn = gr.Button('Free paint engine')
                free_all_btn = gr.Button('Free everything', variant='stop')
                refresh_btn = gr.Button('Refresh status')
            free_shape_btn.click(vram_free_shape, outputs=vram_text)
            free_paint_btn.click(vram_free_paint, outputs=vram_text)
            free_all_btn.click(vram_free_all, outputs=vram_text)
            refresh_btn.click(vram_status, outputs=vram_text)

clear_output()
print('[UI] Launching ...')
demo.queue(max_size=4, default_concurrency_limit=2)
demo.load(lambda: gr.Info("Welcome to Hunyuan3D. Pick a model variant in the Shape tab, drop an image, click Generate."), None)
demo.launch(share=True, show_error=True, height=900, prevent_thread_lock=False)
print('\n[UI] Public URL above. Open it in a new tab.')
print('[UI] The shape engine is loaded on first click in the Shape tab.')

In [ ]:
# @title STEP 5 — Keep-Alive (anti-disconnect)
# @markdown Pings a tiny URL every 60 s to keep the Colab tab active for the 90-min idle limit.

import time, requests, datetime, threading, IPython
_stop = threading.Event()
def _pinger():
    while not _stop.is_set():
        try:
            requests.get('https://huggingface.co/api/models/tencent/Hunyuan3D-2.1', timeout=5)
            IPython.display.clear_output(wait=True)
            print(f'[Keep-Alive] {datetime.datetime.now().strftime("%H:%M:%S")} OK — UI still running above. Stop with `_stop.set()`.')
        except Exception as e:
            print(f'[Keep-Alive] {datetime.datetime.now().strftime("%H:%M:%S")} WARN: {e}')
        _stop.wait(60)
_t = threading.Thread(target=_pinger, daemon=True)
_t.start()
print('[Keep-Alive] Started. Safe to keep this cell running.')

In [ ]:
# @title STEP 6 — Quick Test (single inference smoke test)
# @markdown Runs a single end-to-end shape generation (white mesh only) with the 2.1 3.3 B
# @markdown shape DiT on a downloaded example image. Confirms Step 1 + 2 + 3 are wired correctly.
# @markdown Optional QUICK_DO_PAINT flag exercises the PBR texture pipeline too.

import os, sys, time, pathlib
import torch, trimesh
from PIL import Image
from IPython.display import FileLink, display

REPO_DIR = '/content/Hunyuan3D-2.1'
if REPO_DIR not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, 'hy3dshape'))
    sys.path.insert(0, os.path.join(REPO_DIR, 'hy3dpaint'))
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

QUICK_STEPS = 30
QUICK_OCTREE = 256
QUICK_SEED = 0
QUICK_DO_PAINT = False  # set True to also exercise the PBR texture pipeline

ASSETS_DIR = pathlib.Path('/content/hy3d21_assets')
examples = sorted(ASSETS_DIR.glob('*.png')) if ASSETS_DIR.exists() else []
if not examples:
    raise SystemExit('No example images found. Re-run Step 2 to download them.')
img = Image.open(examples[0]).convert('RGBA')
print(f'[QuickTest] Using {examples[0].name} ({img.size})')

SHAPE.load()
t0 = time.time()
mesh, proc = gen_shape(img, QUICK_STEPS, 5.5, QUICK_SEED, QUICK_OCTREE, 8000, do_rembg=True)
print(f'[QuickTest] shape generation: {time.time() - t0:.1f}s, {mesh.faces.shape[0]} faces')

out_dir = pathlib.Path('/content/hy3d21_runs/quicktest')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'quick_mesh.glb'
mesh.export(str(out_path), include_normals=False)
print(f'[QuickTest] saved {out_path}')
display(FileLink(str(out_path)))

if QUICK_DO_PAINT and _PAINT_IMPORT_OK:
    print('[QuickTest] also running PBR paint ...')
    # Save white mesh as .obj (paint pipeline expects .obj path).
    obj_path = out_dir / 'quick_mesh.obj'
    mesh.export(str(obj_path), include_normals=False)
    PAINT.load()
    t0 = time.time()
    textured_obj = gen_textured(str(obj_path), proc)
    print(f'[QuickTest] PBR paint: {time.time() - t0:.1f}s -> {textured_obj}')
    glb_path = pathlib.Path(textured_obj).with_suffix('.glb')
    if glb_path.exists():
        print(f'[QuickTest] PBR glb: {glb_path}'); display(FileLink(str(glb_path)))
elif QUICK_DO_PAINT:
    print('[QuickTest] QUICK_DO_PAINT=True but PBR paint is not available (custom_rasterizer missing).')

print(f'\n[QuickTest] DONE. Final VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

In [ ]:
# @title STEP 7 — Batch Generation from a .txt list
# @markdown Reads a text file of image paths (one per line) and generates meshes for each.
# @markdown Saves them to a per-run folder. Use this for sweeping a folder of reference images.

import os, sys, time, pathlib, traceback, random
import torch, trimesh
from PIL import Image

REPO_DIR = '/content/Hunyuan3D-2.1'
if REPO_DIR not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, 'hy3dshape'))
    sys.path.insert(0, os.path.join(REPO_DIR, 'hy3dpaint'))
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

# --- Edit these before running ---------------------------------
LIST_FILE = '/content/hy3d21_assets/batch_list.txt'  # one image path per line
BATCH_STEPS = 30
BATCH_CFG = 5.5
BATCH_OCTREE = 256
BATCH_NUM_CHUNKS = 8000
BATCH_SEED = 0
BATCH_DO_REMBG = True
BATCH_DO_PAINT = False  # set True to also PBR-paint each batch item (~21 GB VRAM needed)
OUT_ROOT = pathlib.Path('/content/hy3d21_runs/batch')
MAX_ITEMS = 0  # 0 = no cap
# --------------------------------------------------------------

list_path = pathlib.Path(LIST_FILE)
if not list_path.exists():
    examples = sorted(pathlib.Path('/content/hy3d21_assets').glob('*.png'))
    list_path.write_text('\n'.join(str(p) for p in examples) + '\n')
    print(f'[Batch] No list file found, bootstrapped {len(examples)} examples into {list_path}')

lines = [l.strip() for l in list_path.read_text().splitlines() if l.strip() and not l.startswith('#')]
if MAX_ITEMS: lines = lines[:MAX_ITEMS]
print(f'[Batch] {len(lines)} image(s) queued.')

SHAPE.load()
OUT_ROOT.mkdir(parents=True, exist_ok=True)
batch_dir = OUT_ROOT / f'batch_{int(time.time())}'
batch_dir.mkdir(parents=True, exist_ok=True)

ok = 0; fail = 0
for i, line in enumerate(lines, 1):
    try:
        p = pathlib.Path(line)
        if not p.exists():
            print(f'  [{i:03d}] SKIP (not found): {line}'); fail += 1; continue
        img = Image.open(p).convert('RGBA')
        t0 = time.time()
        mesh, proc = gen_shape(img, BATCH_STEPS, BATCH_CFG, BATCH_SEED,
                                BATCH_OCTREE, BATCH_NUM_CHUNKS, BATCH_DO_REMBG)
        mesh = SHAPE.cleanup(mesh)
        safe = ''.join(c if c.isalnum() else '_' for c in p.stem[:40]).strip('_') or f'item_{i:02d}'
        out_path = batch_dir / f'{i:03d}_{safe}.glb'
        mesh.export(str(out_path), include_normals=False)
        elapsed = time.time() - t0
        if BATCH_DO_PAINT and _PAINT_IMPORT_OK:
            try:
                obj_path = batch_dir / f'{i:03d}_{safe}.obj'
                mesh.export(str(obj_path), include_normals=False)
                PAINT.load()
                textured_obj = PAINT.texture(str(obj_path), proc)
                tex_glb = pathlib.Path(textured_obj).with_suffix('.glb')
                print(f'  [{i:03d}] {p.name} -> {out_path.name} + {tex_glb.name} ({elapsed:.1f}s, shape+paint)')
            except Exception as e:
                print(f'  [{i:03d}] {p.name} -> {out_path.name} (shape OK; paint failed: {e})')
        else:
            print(f'  [{i:03d}] {p.name} -> {out_path.name} ({elapsed:.1f}s)')
        ok += 1
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except Exception as e:
        print(f'  [{i:03d}] EXCEPTION on {line}: {e}'); traceback.print_exc(); fail += 1

print(f'\n[Batch] DONE: {ok} OK / {fail} failed / {len(lines)} total -> {batch_dir}')